# SMS Spam & Phishing Detector

Classify English SMS as `ham` or `spam`, and measure how much regex fingerprint features add on top of a bag-of-words model. Specs: `docs/project-specs.md`.

## 0. Setup

Imports and the fixed random seed used everywhere (NFR-1).

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_PATH = "data/spam.csv"

pd.set_option("display.max_colwidth", 120)

## 1. Load and deduplicate (FR-1)

Read the CSV with `latin-1` (its real encoding), keep only the label and text columns, and add a numeric target `y`.

In [2]:
raw = pd.read_csv(DATA_PATH, encoding="latin-1")
print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))

df = raw[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
df["y"] = (df["label"] == "spam").astype(int)
df.head()

Raw shape: (5572, 5)
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


,label,text,y
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives around here though",0


A message that appeared with both labels would be contradictory training data; this check documents that the dataset has none.

In [3]:
labels_per_text = df.groupby(df["text"].str.strip())["label"].nunique()
assert (labels_per_text == 1).all(), "Some texts appear with both labels"
print("No message has conflicting labels.")

No message has conflicting labels.


Drop duplicates before any split so the same message never lands in both train and test. Duplicates are compared on trimmed text, but the `text` column itself stays raw for the regex branch.

In [4]:
n_before = len(df)
df = (df.loc[~df["text"].str.strip().duplicated(keep="first")]
        .reset_index(drop=True))

print(f"Rows before: {n_before}, after: {len(df)}, removed: {n_before - len(df)}")
print("\nClass balance after deduplication:")
print(df["label"].value_counts().to_frame("count")
        .assign(share=lambda t: (t["count"] / t["count"].sum()).round(3)))

Rows before: 5572, after: 5158, removed: 414

Class balance after deduplication:
       count  share
label              
ham     4516  0.876
spam     642  0.124
